In [ ]:
!pip install easyocr opencv-python-headless tensorflow numpy

In [ ]:
from google.colab import files

uploaded = files.upload()

import numpy as np
import tensorflow as tf

MODEL_PATH = "recruitment_detector.tflite"
VOCAB_PATH = "vocab.txt"
LABELS_PATH = "labels.txt"
MAX_LEN = 60

def load_vocab(path):
    vocab = {}
    with open(path, "r", encoding="utf-8") as f:
        for index, token in enumerate(f.read().splitlines()):
            vocab[token] = index
    return vocab

def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()

def text_to_sequence(text, vocab, max_len=60):
    text = text.lower()
    words = text.split()

    sequence = []
    for word in words:
        token_id = vocab.get(word, vocab.get("[UNK]", 1))
        sequence.append(token_id)

    if len(sequence) < max_len:
        sequence += [0] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]

    return np.array([sequence], dtype=np.int64)

vocab = load_vocab(VOCAB_PATH)
labels = load_labels(LABELS_PATH)

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_text(text):
    input_data = text_to_sequence(text, vocab, MAX_LEN)

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]["index"])
    score = float(output[0][0])

    label = "reclutamiento" if score >= 0.5 else "safe"

    return label, score

In [ ]:
import easyocr

from google.colab import files
import easyocr

uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]

reader = easyocr.Reader(["es", "en"])
result = reader.readtext(IMAGE_PATH)

detected_text = " ".join([item[1] for item in result])

print("Texto detectado:")
print(detected_text)

label, score = predict_text(detected_text)

print("\nPredicción:")
print(label, score)